In [ ]:
# ML-Inter
# excercise-3 ( handeling categorical variables(values) / columns )

import pandas as pd

from sklearn.model_selection import train_test_split

data = pd.read_csv(file_path)

y = data.pred_target_col

X = data.drop(['pred_target_col'] , axis = 1)

X_train , X_val , y_train , y_val = train_test_split( X , y , train_size = 0.8 , test_size = 0.2 , random_state = 0 )


# method-1 drop categorical columns

cols_with_category = [ col for col in data.columns if data[col].dtypes == 'object']
data = data.drop( cols_with_category , axis = 1 )


# method-2 ordinal encoding ( replacing a categorical column with numbers ) , this is done for columns containig ordinal variables ( variables that has some intrinstic order {
# 'worse' < 'bad' < 'better' < 'good'} )

from sklearn.preprocessing import OrdinalEncoder

ordinal_encode = OrdinalEncoder()
X_train[cols_with_category] = ordinal_encode.fit_transform(X_train[cols_with_category])
X_val[cols_with_category] = ordinal_encode.transform(X_val[cols_with_category])  


# method-3 One Hot Encoding ( replacing categorical column with many columns , each containing one unique value from the replaced column : as it's name)
# therefore the new columns will have 1 : if one unique value exists in that row and 0 : if not 
# cardinality = no_of unique values in a column 
# this method is used when there is no ordinal variable , rather nominal variable ( no implicit ordering can be made {'red' , 'yellow' , 'blue'})
# one hot encoding is not used for categorical columns with more than 15 : cardinality ( it just don't work better )

from sklearn.preprocessing import OneHotEncoder

OH_encoder = OneHotEncoder( handle_unknown = 'ignore' , sparse = False) # here handle_unknows is used because when the validation data columns have some values that were not in the training data
# here OneHotEncoder class return as sparse matrix rather than a numpy array 

OH_X_train = pd.DataFrame(OH_encoder.fit_transform(X_train[cols_with_category]))
OH_X_val = pd.DataFrame(OH_encoder.transform(X_val[cols_with_category]))

# OH encoding removes index ; add them back 

OH_X_train.index = X_train.index
OH_X_val.index = X_val.index

# drop the existing categorical columns ( on which we just applied one hot encoding )
X_train = X_train.drop(cols_with_category , axis =1)
X_val = X_val.drop(cols_with_category , axis = 1)

# add them back after OneHotEncoding
OH_X_train = pd.concat([X_train , OH_X_train])
OH_X_val = pd.concat([X_val , OH_X_val])

# now make sure both text and train data columns have the same data_type : str ( for fitting OneHotEncoded data ; it requires all the columns to have the same type ,
# cuz if one column in train is str and the same column in test is object/int/float , then one hot encoding give 'Found unknown categories during transform' error )

OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_val.columns = OH_X_val.columns.astype(str)


